# BPF on Kaggle: End-to-End Notebook

This notebook prepares environment, maps Pascal VOC from Kaggle input, patches YAML configs for writable outputs, runs BPF stages, and visualizes predictions.

In [ ]:
import os
import sys
import yaml
import json
import random
import shutil
import subprocess
from pathlib import Path

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def run_cmd(cmd: str, allow_fail: bool = False):
    print(f'\n[CMD] {cmd}')
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f'Command failed ({result.returncode}): {cmd}')
    return result.returncode

REPO_URL = 'https://github.com/HelcurtLordno1/Continual-Learning-Bridge-Past-and-Future.git'
WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'Continual-Learning-Bridge-Past-and-Future'

WORKDIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    run_cmd(f'git -C {REPO_DIR} pull', allow_fail=True)
else:
    run_cmd(f'git clone {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
print('Current directory:', Path.cwd())

In [ ]:
try:
    run_cmd('python -m pip install --upgrade pip')
    run_cmd('pip install yacs ninja torchvision torchmetrics')
    run_cmd('pip install -r requirements.txt')

    if Path('setup.py').exists():
        try:
            # First try normal extension build.
            run_cmd('python setup.py build develop')
        except Exception as build_exc:
            print('Normal extension build failed. Falling back to no-extension mode...')
            print(build_exc)
            run_cmd('MASKRCNN_BENCHMARK_NO_EXT=1 python setup.py build develop')
    else:
        print('setup.py not found, skipping extension build.')
except Exception as exc:
    raise RuntimeError(f'Dependency/build step failed: {exc}')

In [ ]:
KAGGLE_VOC_ROOT = Path('/kaggle/input/voc0712')
VOCDEVKIT_SRC = KAGGLE_VOC_ROOT / 'VOCdevkit'
if not VOCDEVKIT_SRC.exists():
    raise FileNotFoundError(f'Missing dataset path: {VOCDEVKIT_SRC}')

datasets_dir = REPO_DIR / 'datasets'
datasets_dir.mkdir(parents=True, exist_ok=True)
datasets_voc_link = datasets_dir / 'VOCdevkit'

data_voc07_dir = REPO_DIR / 'data' / 'voc07'
data_voc07_dir.mkdir(parents=True, exist_ok=True)
voc07_link = data_voc07_dir / 'VOCdevkit'

for link_path in [datasets_voc_link, voc07_link]:
    if link_path.is_symlink() or link_path.exists():
        if link_path.is_symlink() and os.path.realpath(link_path) == str(VOCDEVKIT_SRC):
            print(f'Symlink already valid: {link_path}')
        else:
            if link_path.is_dir() and not link_path.is_symlink():
                shutil.rmtree(link_path)
            else:
                link_path.unlink()
            os.symlink(VOCDEVKIT_SRC, link_path)
            print(f'Re-linked: {link_path} -> {VOCDEVKIT_SRC}')
    else:
        os.symlink(VOCDEVKIT_SRC, link_path)
        print(f'Linked: {link_path} -> {VOCDEVKIT_SRC}')

required = [
    voc07_link / 'VOC2007' / 'JPEGImages',
    voc07_link / 'VOC2007' / 'Annotations',
    voc07_link / 'VOC2007' / 'ImageSets' / 'Main',
]
for p in required:
    if not p.exists():
        raise FileNotFoundError(f'Required VOC folder missing: {p}')

print('Dataset symlink mapping complete.')

In [ ]:
from torchvision.datasets import VOCSegmentation

VOC_COLORMAP = [
    [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128], [128, 0, 128],
    [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0], [64, 128, 0], [192, 128, 0],
    [64, 0, 128], [192, 0, 128], [64, 128, 128], [192, 128, 128], [0, 64, 0], [128, 64, 0],
    [0, 192, 0], [128, 192, 0], [0, 64, 128],
]

class PascalVOCSearchDataset(VOCSegmentation):
    def __init__(self, root='/kaggle/input/voc0712', year='2007', image_set='train', download=False, transform=None):
        super().__init__(root=root, year=year, image_set=image_set, download=download, transform=transform)

    @staticmethod
    def _convert_to_segmentation_mask(mask):
        h, w = mask.shape[:2]
        seg = np.zeros((h, w, len(VOC_COLORMAP)), dtype=np.float32)
        for idx, color in enumerate(VOC_COLORMAP):
            seg[:, :, idx] = np.all(mask == color, axis=-1).astype(float)
        return seg

    def __getitem__(self, index):
        image = cv2.imread(self.images[index])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.masks[index])
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
        mask = self._convert_to_segmentation_mask(mask)
        return image, mask.argmax(axis=2).astype(np.int64)

ds = PascalVOCSearchDataset(root='/kaggle/input/voc0712', year='2007', image_set='train')
img, msk = ds[0]
print('Dataset check OK:', img.shape, msk.shape)

In [ ]:
def patch_yaml_for_kaggle(src_yaml: Path, dst_yaml: Path, output_dir: str, tensorboard_dir: str, max_iter=None, ims_per_batch=None, checkpoint_period=None):
    if not src_yaml.exists():
        raise FileNotFoundError(f'Missing source yaml: {src_yaml}')

    with src_yaml.open('r', encoding='utf-8') as f:
        data = yaml.safe_load(f)

    data['OUTPUT_DIR'] = output_dir
    data['TENSORBOARD_DIR'] = tensorboard_dir
    solver = data.get('SOLVER', {})
    if max_iter is not None:
        solver['MAX_ITER'] = int(max_iter)
    if ims_per_batch is not None:
        solver['IMS_PER_BATCH'] = int(ims_per_batch)
    if checkpoint_period is not None:
        solver['CHECKPOINT_PERIOD'] = int(checkpoint_period)
    data['SOLVER'] = solver

    dst_yaml.parent.mkdir(parents=True, exist_ok=True)
    with dst_yaml.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False)
    return data

generated_cfg_dir = REPO_DIR / 'configs' / 'kaggle_runtime'
generated_cfg_dir.mkdir(parents=True, exist_ok=True)

QUICK_RUN = True
max_iter_val = 500 if QUICK_RUN else None
ims_val = 2 if QUICK_RUN else None

src_stage1 = REPO_DIR / 'configs/OD_cfg/10-10/e2e_faster_rcnn_R_50_C4_4x_att1_rpn1.yaml'
src_stage2 = REPO_DIR / 'configs/OD_cfg/10-10/e2e_faster_rcnn_R_50_C4_4x_sec_10.yaml'
src_stage3 = REPO_DIR / 'configs/OD_cfg/10-10/e2e_faster_rcnn_R_50_C4_4x_BPF_Target_model.yaml'

cfg_stage1 = generated_cfg_dir / 'stage1_10_10.yaml'
cfg_stage2 = generated_cfg_dir / 'stage2_10_10.yaml'
cfg_stage3 = generated_cfg_dir / 'stage3_10_10.yaml'

patch_yaml_for_kaggle(src_stage1, cfg_stage1, '/kaggle/working/output/10-10/stage1', '/kaggle/working/output/tb/stage1', max_iter=max_iter_val, ims_per_batch=ims_val, checkpoint_period=200)
patch_yaml_for_kaggle(src_stage2, cfg_stage2, '/kaggle/working/output/10-10/stage2', '/kaggle/working/output/tb/stage2', max_iter=max_iter_val, ims_per_batch=ims_val, checkpoint_period=200)

stage3_data = patch_yaml_for_kaggle(src_stage3, cfg_stage3, '/kaggle/working/output/10-10/stage3', '/kaggle/working/output/tb/stage3', max_iter=max_iter_val, ims_per_batch=ims_val, checkpoint_period=200)
stage3_data['MODEL']['WEIGHT'] = '/kaggle/working/output/10-10/stage1/model_trimmed.pth'
stage3_data['MODEL']['SOURCE_WEIGHT'] = '/kaggle/working/output/10-10/stage1/model_trimmed.pth'
stage3_data['MODEL']['FINETUNE_WEIGHT'] = '/kaggle/working/output/10-10/stage2/model_final.pth'
with cfg_stage3.open('w', encoding='utf-8') as f:
    yaml.safe_dump(stage3_data, f, sort_keys=False)

print('Generated runtime YAMLs:', cfg_stage1, cfg_stage2, cfg_stage3)

## Main Training
Runs stage-1, trim, stage-2, and stage-3 in sequence.

In [ ]:
os.chdir(REPO_DIR)

RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_STAGE3 = True

if RUN_STAGE1:
    run_cmd(f'python tools/train_first_step.py -c {cfg_stage1} --skip-test')
    run_cmd("python tools/trim_detectron_model.py --name '10-10/stage1'", allow_fail=True)
    src_trim = Path('/kaggle/working/output/10-10/stage1/model_trimmed.pth')
    if not src_trim.exists():
        src_final = Path('/kaggle/working/output/10-10/stage1/model_final.pth')
        if src_final.exists():
            shutil.copy2(src_final, src_trim)

if RUN_STAGE2:
    run_cmd(f'python tools/train_first_step.py -c {cfg_stage2} --skip-test')

if RUN_STAGE3:
    original_stage3 = REPO_DIR / 'configs/OD_cfg/10-10/e2e_faster_rcnn_R_50_C4_4x_BPF_Target_model.yaml'
    backup_stage3 = original_stage3.with_suffix('.yaml.bak')
    shutil.copy2(original_stage3, backup_stage3)
    shutil.copy2(cfg_stage3, original_stage3)
    try:
        run_cmd('python tools/train_incremental_finetune_all.py -t 10-10 -n kaggle --cls 0.15 -l 0.4 -high 0.7 -lw 1.0 -hw 0.3 --skip-test')
    finally:
        shutil.copy2(backup_stage3, original_stage3)
        if backup_stage3.exists():
            backup_stage3.unlink()

print('Training pipeline finished.')

In [ ]:
target_model = Path('/kaggle/working/output/10-10/stage3/model_final.pth')
if target_model.exists():
    run_cmd(f'python tools/test_net.py --config-file {cfg_stage3} MODEL.WEIGHT {target_model}', allow_fail=True)
else:
    print('Stage-3 model not found yet. Skip test_net.')

In [ ]:
from PIL import Image
import matplotlib.patches as patches

class UnNormalize:
    def __init__(self, mean=(102.9801, 115.9465, 122.7717), std=(1.0, 1.0, 1.0)):
        self.mean = np.array(mean, dtype=np.float32)
        self.std = np.array(std, dtype=np.float32)

    def __call__(self, chw_bgr):
        img = chw_bgr.copy()
        for c in range(3):
            img[c] = img[c] * self.std[c] + self.mean[c]
        return img[::-1].transpose(1, 2, 0).clip(0, 255).astype(np.uint8)

unorm = UnNormalize()

from maskrcnn_benchmark.config import cfg as det_cfg
from maskrcnn_benchmark.modeling.detector import build_detection_model
from maskrcnn_benchmark.utils.checkpoint import DetectronCheckpointer

cfg_vis = det_cfg.clone()
cfg_vis.merge_from_file(str(cfg_stage3))

ckpt_candidates = [
    Path('/kaggle/working/output/10-10/stage3/model_final.pth'),
    Path('/kaggle/working/output/10-10/stage1/model_final.pth'),
]
ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)
if ckpt_path is None:
    raise FileNotFoundError('No checkpoint found for visualization.')

cfg_vis.MODEL.WEIGHT = str(ckpt_path)
cfg_vis.freeze()

model = build_detection_model(cfg_vis)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
_ = DetectronCheckpointer(cfg_vis, model).load(cfg_vis.MODEL.WEIGHT)

voc_test_images = Path('/kaggle/input/voc0712/VOCdevkit/VOC2007/JPEGImages')
image_files = sorted(voc_test_images.glob('*.jpg'))[:3]
if len(image_files) == 0:
    raise FileNotFoundError(f'No test images found in {voc_test_images}')

fig, axes = plt.subplots(1, len(image_files), figsize=(6 * len(image_files), 6))
if len(image_files) == 1:
    axes = [axes]

for ax, img_path in zip(axes, image_files):
    rgb = np.array(Image.open(img_path).convert('RGB'))
    bgr = rgb[:, :, ::-1].astype(np.float32)
    chw = bgr.transpose(2, 0, 1)

    chw_norm = chw.copy()
    for c in range(3):
        chw_norm[c] = (chw_norm[c] - cfg_vis.INPUT.PIXEL_MEAN[c]) / cfg_vis.INPUT.PIXEL_STD[c]

    tensor = torch.from_numpy(chw_norm).float().to(device)

    with torch.no_grad():
        preds = model([tensor])[0].to('cpu')

    ax.imshow(unorm(chw_norm))
    ax.set_title(img_path.name)
    ax.axis('off')

    boxes = preds.bbox.numpy() if len(preds) > 0 else np.zeros((0, 4))
    scores = preds.get_field('scores').numpy() if len(preds) > 0 else np.array([])
    labels = preds.get_field('labels').numpy() if len(preds) > 0 else np.array([])

    keep = scores >= 0.6
    for box, score, label in zip(boxes[keep], scores[keep], labels[keep]):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1, f'cls:{int(label)} {score:.2f}', color='yellow', fontsize=9, bbox=dict(facecolor='black', alpha=0.5))

plt.tight_layout()
plt.show()
print('Visualization complete from', ckpt_path)